# Notebook 01 — Exploratory Data Analysis

**Project:** ML-Based Malaria Occurrence Prediction System  
**Dataset:** `Malaria_Dataset.csv` — 1,622 patients, binary clinical outcome  
**Purpose:** Understand the data distribution, check class balance, and identify key predictors.


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.malaria_forecast.data_loader import load_raw_dataset
from src.malaria_forecast.features import ALL_NUMERIC_FEATURES, TARGET_COLUMN

sns.set_theme(style='whitegrid', palette='Set2')
%matplotlib inline


## 1. Load Dataset

In [ ]:
df = load_raw_dataset('../dataset/Malaria_Dataset.csv')
print('Shape:', df.shape)
df.head()

## 2. Basic Overview

In [ ]:
print('=== Data Types ===')
print(df.dtypes)

print('\n=== Missing Values ===')
print(df.isnull().sum())

print('\n=== Descriptive Statistics ===')
df.describe()

## 3. Target Distribution

In [ ]:
counts = df[TARGET_COLUMN].value_counts().sort_index()
labels = ['Negative (0)', 'Positive (1)']
colors = ['#4CAF50', '#F44336']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(labels, counts.values, color=colors, edgecolor='black', width=0.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')
axes[0].set_title('Malaria Occurrence Distribution')
axes[0].set_ylabel('Count')
axes[1].pie(counts.values, labels=[f'{l}\n({v})' for l, v in zip(labels, counts.values)],
            colors=colors, autopct='%1.1f%%', startangle=140)
axes[1].set_title('Class Proportion')
plt.suptitle('Target Variable Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nClass balance: {counts[1]/(counts[0]+counts[1]):.1%} positive / {counts[0]/(counts[0]+counts[1]):.1%} negative')

## 4. Correlation Heatmap

In [ ]:
numeric_cols = ALL_NUMERIC_FEATURES + [TARGET_COLUMN]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr, ax=ax, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, square=True, annot_kws={'size': 8})
ax.set_title('Feature Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Distributions

In [ ]:
features = ALL_NUMERIC_FEATURES
n_cols = 4
n_rows = -(-len(features) // n_cols)  # ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
axes = axes.flatten()

symptom_cols = [c for c in features if c not in ('age', 'length_of_stay')]

for i, col in enumerate(features):
    ax = axes[i]
    if col in symptom_cols:
        counts_col = df[col].value_counts().sort_index()
        ax.bar(['No (0)', 'Yes (1)'], counts_col.values, color=['#78C8E0', '#FF8A65'], edgecolor='black')
    else:
        ax.hist(df[col].dropna(), bins=20, color='#5C85D6', edgecolor='black', alpha=0.85)
    ax.set_title(col.replace('_', ' ').title(), fontsize=9, fontweight='bold')
    ax.tick_params(labelsize=7)

for j in range(len(features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Symptom Prevalence by Outcome

In [ ]:
symptom_features = [
    'fever', 'headache', 'abdominal_pain', 'general_body_malaise',
    'dizziness', 'vomiting', 'confusion', 'backache',
    'chest_pain', 'coughing', 'joint_pain'
]

pos_rates = df[df[TARGET_COLUMN]==1][symptom_features].mean() * 100
neg_rates = df[df[TARGET_COLUMN]==0][symptom_features].mean() * 100

x = range(len(symptom_features))
labels = [s.replace('_', '\n').title() for s in symptom_features]

fig, ax = plt.subplots(figsize=(15, 6))
ax.bar([i - 0.2 for i in x], neg_rates, 0.38, label='Negative', color='#4CAF50', edgecolor='black')
ax.bar([i + 0.2 for i in x], pos_rates, 0.38, label='Positive', color='#F44336', edgecolor='black')
ax.set_xticks(list(x))
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('Patients with Symptom (%)')
ax.set_title('Symptom Prevalence by Malaria Outcome', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/boxplot_symptoms_by_target.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Key Observations

- **Class imbalance:** ~72.6% positive, 27.4% negative — `class_weight='balanced'` applied in all models.
- **No missing values:** Dataset is complete (0 nulls across all 1,622 rows).
- **Length of stay:** Engineered from admission/discharge dates — ranges 1–10 days.
- **Symptom prevalence:** Several symptoms (e.g. General Body Malaise, Backache) appear highly prevalent in both classes.
- **Correlation:** Most binary symptom flags show low inter-correlation — each is independently informative.
